# RoboQuest2026 Tier1 Official-Compatible GA

公式Tier1ノートブックと同じ逃げ学習フローを、遺伝的アルゴリズムで自動反復します。
環境本体・観測空間・行動空間・PPO基本設定は公式Tier1互換を優先し、探索対象は公式スライダー相当の値に絞ります。

最初の個体には必ず公式デフォルト設定を入れるため、本家Tier1相当の動作確認にも使えます。


In [ ]:
# セットアップ
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, json, time, random, hashlib, shutil, csv
from pathlib import Path

REPO = Path('/content/RoboQuest2026')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/SingularityBattleQuest/RoboQuest2026.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', 'origin', 'main'], check=False)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
# ColabでMuJoCoをヘッドレス実行するための設定。学習だけでも安全側で入れておきます。
_has_gpu = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0
if _has_gpu:
    os.environ['MUJOCO_GL'] = 'egl'
else:
    subprocess.run('apt-get update -qq && apt-get install -y -q libosmesa6', shell=True, check=False)
    os.environ['MUJOCO_GL'] = 'osmesa'
# GA探索ではビューアーは使わないため、必要最小限の学習依存だけを入れます。
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'mujoco', 'gymnasium', 'stable-baselines3[extra]',
                'numpy', 'pandas', 'matplotlib', 'tqdm', 'rich',
                'imageio', 'imageio-ffmpeg'], check=False)
print('setup done')

# Tier1/Tier2共通の強い歩行ベースを使うためのColabパッチ。
# 1. Tag環境も歩行学習と同じ go2_posctrl.xml を使う。
# 2. 高レベル逃げ命令は、低レベル歩行が得意な前進+軽い旋回へ写像する。
def patch_colab_tier1_environment(repo):
    repo = Path(repo)
    arena_posctrl = repo / 'models/go2/arena_posctrl.xml'
    arena_posctrl.write_text("""<mujoco model="go2_arena_posctrl">
  <include file="go2_posctrl.xml"/>
  <statistic center="0 0 0.3" extent="4"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0 0 0"/>
    <rgba haze="0.1 0.15 0.2 1"/>
    <global azimuth="120" elevation="-30"/>
  </visual>
  <asset>
    <texture type="2d" name="floor_tex" builtin="checker" rgb1="0.35 0.55 0.35" rgb2="0.28 0.45 0.28" width="256" height="256"/>
    <material name="floor_mat" texture="floor_tex" texrepeat="5 5" reflectance="0.1"/>
    <material name="wall_mat" rgba="0.55 0.55 0.65 1"/>
    <material name="oni_mat"  rgba="0.95 0.12 0.12 1"/>
  </asset>
  <worldbody>
    <light name="main_light" pos="0 0 5" dir="0 0 -1" directional="true" diffuse="0.8 0.8 0.8" specular="0.1 0.1 0.1"/>
    <light name="fill_light" pos="0 0 3" dir="0 1 -1" directional="true" diffuse="0.3 0.3 0.35" specular="0 0 0"/>
    <geom name="floor" type="plane" size="2.5 2.5 0.1" material="floor_mat" contype="1" conaffinity="1" friction="0.8 0.1 0.1"/>
    <geom name="wall_n" type="box" pos="0  2.55 0.5" size="2.6 0.05 0.5" material="wall_mat" contype="1" conaffinity="1"/>
    <geom name="wall_s" type="box" pos="0 -2.55 0.5" size="2.6 0.05 0.5" material="wall_mat" contype="1" conaffinity="1"/>
    <geom name="wall_e" type="box" pos=" 2.55 0 0.5" size="0.05 2.6 0.5" material="wall_mat" contype="1" conaffinity="1"/>
    <geom name="wall_w" type="box" pos="-2.55 0 0.5" size="0.05 2.6 0.5" material="wall_mat" contype="1" conaffinity="1"/>
    <camera name="top_view" pos="0 0 9" euler="0 0 0" fovy="60"/>
    <camera name="side_view" pos="6 -4 4" euler="55 0 45" fovy="50"/>
    <body name="oni" pos="0 0 0.3">
      <joint name="oni_x" type="slide" axis="1 0 0" limited="true" range="-2.4 2.4" damping="100" frictionloss="0"/>
      <joint name="oni_y" type="slide" axis="0 1 0" limited="true" range="-2.4 2.4" damping="100" frictionloss="0"/>
      <geom name="oni_body" type="sphere" size="0.28" material="oni_mat" contype="0" conaffinity="0" mass="0.001"/>
      <geom name="oni_eye" type="sphere" size="0.08" pos="0.25 0 0.1" rgba="1 1 1 1" contype="0" conaffinity="0" mass="0"/>
    </body>
  </worldbody>
  <keyframe>
    <key name="arena_home" qpos="0 0 0.27 1 0 0 0 0 0.9 -1.8 0 0.9 -1.8 0 0.9 -1.8 0 0.9 -1.8 1.6 1.6" ctrl="0 0.9 -1.8 0 0.9 -1.8 0 0.9 -1.8 0 0.9 -1.8"/>
  </keyframe>
</mujoco>
""", encoding='utf-8')

    tag_env_path = repo / 'roboquest/envs/go2_tag_env.py'
    tag_text = tag_env_path.read_text(encoding='utf-8')
    tag_text = tag_text.replace('ARENA_XML = os.path.join(_MODEL_DIR, "arena.xml")',
                                'ARENA_XML = os.path.join(_MODEL_DIR, "arena_posctrl.xml")')
    tag_env_path.write_text(tag_text, encoding='utf-8')

    hier_path = repo / 'roboquest/envs/go2_tag_hierarchical_env.py'
    hier_text = hier_path.read_text(encoding='utf-8')
    changed = False
    if 'high_level_command_mode: str = "direct"' in hier_text:
        hier_text = hier_text.replace('high_level_command_mode: str = "direct"',
                                      'high_level_command_mode: str = "safe_forward"')
        changed = True
    old_ranges = '''                "vx": (0.25, 0.55),
                "vy": (-0.05, 0.05),
                "omega": (-0.20, 0.20),'''
    new_ranges = '''                "vx": (0.75, 1.55),
                "vy": (-0.02, 0.02),
                "omega": (-0.95, 0.95),'''
    if old_ranges in hier_text:
        hier_text = hier_text.replace(old_ranges, new_ranges)
        changed = True
    if changed:
        hier_path.write_text(hier_text, encoding='utf-8')
        print('Tier1 environment patch: arena_posctrl + race-speed safe_forward defaults applied')
    else:
        print('Tier1 environment patch: arena_posctrl applied. race-speed defaults were already applied or not found.')

patch_colab_tier1_environment(REPO)


In [ ]:
# 実験設定
team_name = 'team_ga' #@param {type:'string'}
run_name = 'tier1_official_full_001' #@param {type:'string'}
tier1_profile = 'official_full' #@param ['quick_test', 'official_smoke', 'official_full', 'search_full', 'overnight', 'custom']

# custom を選んだ場合だけ、下の手動値を使います。
manual_population_size = 4 #@param {type:'integer'}
manual_generations = 2 #@param {type:'integer'}
manual_elite_count = 1 #@param {type:'integer'}
manual_flee_timesteps_choices = [200000] #@param
manual_flee_num_envs_choices = [2] #@param
manual_eval_seeds = [100, 101, 102, 103, 104] #@param
show_ppo_progress = True #@param {type:'boolean'}


# 歩行ベースの選択。
# drive_zip: PCで作った強い歩行ベースzipをDriveから読む。推奨。
# repo_pretrained: GitHub同梱のsmooth_walkを使う。比較・緊急用。
walk_source_mode = 'drive_zip' #@param ['drive_zip', 'repo_pretrained']
walk_bundle_zip = '/content/drive/MyDrive/RoboQuest2026_GA/walk_bases/tier1_walk_speed_race_001_bundle.zip' #@param {type:'string'}

mutation_rate = 0.25 #@param {type:'number'}
crossover_rate = 0.65 #@param {type:'number'}
max_runtime_hours = 10.5 #@param {type:'number'}
seed = 0 #@param {type:'integer'}

# 公式Tier1の標準値。最初の個体として必ず評価します。
OFFICIAL_DEFAULT = dict(
    survival_weight=0.5,
    distance_weight=1.0,
    tag_penalty=50.0,
    oni_speed=0.025,
    flee_timesteps=200000,
    flee_num_envs=2,
)

PROFILE_SETTINGS = {
    # 動作確認用。強さは見ません。
    'quick_test': dict(population_size=2, generations=1, elite_count=1,
                       flee_timesteps_choices=[10000], flee_num_envs_choices=[1],
                       eval_seeds=[100, 101]),
    # 公式フローが落ちずに動くかを見る軽量版。
    'official_smoke': dict(population_size=2, generations=1, elite_count=1,
                           flee_timesteps_choices=[50000], flee_num_envs_choices=[1],
                           eval_seeds=[100, 101, 102]),
    # 本家Tier1に近い本番基準。まずはこれを推奨。
    'official_full': dict(population_size=4, generations=2, elite_count=1,
                          flee_timesteps_choices=[200000], flee_num_envs_choices=[2],
                          eval_seeds=[100, 101, 102, 103, 104]),
    # 公式範囲内で広げる探索。
    'search_full': dict(population_size=6, generations=3, elite_count=2,
                        flee_timesteps_choices=[150000, 200000, 300000], flee_num_envs_choices=[1, 2],
                        eval_seeds=[100, 101, 102, 103, 104]),
    # 放置用。Colabの切断に注意。
    'overnight': dict(population_size=8, generations=5, elite_count=2,
                      flee_timesteps_choices=[200000, 300000, 500000], flee_num_envs_choices=[1, 2],
                      eval_seeds=[100, 101, 102, 103, 104, 105, 106, 107, 108, 109]),
}
if tier1_profile == 'custom':
    active_profile = dict(population_size=manual_population_size, generations=manual_generations,
                          elite_count=manual_elite_count,
                          flee_timesteps_choices=list(manual_flee_timesteps_choices),
                          flee_num_envs_choices=list(manual_flee_num_envs_choices),
                          eval_seeds=list(manual_eval_seeds))
else:
    active_profile = PROFILE_SETTINGS[tier1_profile]

population_size = int(active_profile['population_size'])
generations = int(active_profile['generations'])
elite_count = int(active_profile['elite_count'])
flee_timesteps_choices = list(active_profile['flee_timesteps_choices'])
flee_num_envs_choices = list(active_profile['flee_num_envs_choices'])
eval_seeds = tuple(int(s) for s in active_profile['eval_seeds'])

BASE_DIR = Path('/content/drive/MyDrive/RoboQuest2026_GA/tier1') / run_name
BASE_DIR.mkdir(parents=True, exist_ok=True)
print('profile:', tier1_profile, active_profile)
print('folder:', BASE_DIR)
print('公式デフォルト個体:', OFFICIAL_DEFAULT)


In [ ]:
# 共通関数（公式Tier1互換）
from dataclasses import asdict
from stable_baselines3 import PPO
from scripts.notebook_workflow import train_policy, evaluate_flee, load_bundled_walk, FLEE_PPO
from roboquest.utils.reward_utils import FleeRewardConfig

rng = random.Random(seed)

# Tier1では公式スライダー相当だけを探索します。
# PPO構造や環境本体は公式Tier1互換を優先して固定します。
SPACE = {
    'survival_weight': ('uniform', 0.0, 2.0),
    'distance_weight': ('uniform', 0.0, 3.0),
    'tag_penalty': ('uniform', 10.0, 100.0),
    'oni_speed': ('uniform', 0.01, 0.05),
    'flee_timesteps': ('choice', flee_timesteps_choices),
    'flee_num_envs': ('choice', flee_num_envs_choices),
}


def sample_one(key, spec):
    kind = spec[0]
    if kind == 'uniform':
        return rng.uniform(float(spec[1]), float(spec[2]))
    if kind == 'choice':
        return rng.choice(list(spec[1]))
    raise ValueError(spec)


def normalize_individual(ind):
    out = dict(OFFICIAL_DEFAULT)
    out.update(ind)
    out['flee_timesteps'] = int(out['flee_timesteps'])
    out['flee_num_envs'] = int(out['flee_num_envs'])
    return out


def sample_individual():
    return normalize_individual({k: sample_one(k, v) for k, v in SPACE.items()})


def make_initial_population():
    # 最初は必ず公式デフォルトから始めます。
    pop = [normalize_individual(OFFICIAL_DEFAULT)]
    while len(pop) < population_size:
        pop.append(sample_individual())
    return pop[:population_size]


def hash_individual(ind):
    normalized = normalize_individual(ind)
    s = json.dumps(normalized, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(s.encode()).hexdigest()[:12]


def mutate(ind):
    child = normalize_individual(ind)
    for k, spec in SPACE.items():
        if rng.random() < mutation_rate:
            child[k] = sample_one(k, spec)
    return normalize_individual(child)


def crossover(a, b):
    a = normalize_individual(a)
    b = normalize_individual(b)
    if rng.random() > crossover_rate:
        return dict(a)
    return normalize_individual({k: (a[k] if rng.random() < 0.5 else b[k]) for k in a})


def score_results(results):
    survived = [float(r['survived_seconds']) for r in results]
    mean_survived = sum(survived) / len(survived)
    worst = min(survived)
    escaped = sum(1 for r in results if r.get('escaped'))
    tagged = sum(1 for r in results if r.get('tagged'))
    fell = sum(1 for r in results if r.get('fell'))
    mean_distance = sum(float(r.get('mean_distance', 0)) for r in results) / len(results)
    # 逃げ切りを最優先。転倒は強く減点。距離は補助。
    score = mean_survived + escaped * 40 + worst * 0.35 + mean_distance * 2 - tagged * 12 - fell * 35
    return dict(score=score, mean_survived_seconds=mean_survived, worst_survived_seconds=worst,
                escaped_count=escaped, tagged_count=tagged, fell_count=fell, mean_distance=mean_distance)


def append_jsonl(path, row):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')


def prepare_walk_source():
    source_dir = BASE_DIR / 'walk_source'
    source_dir.mkdir(parents=True, exist_ok=True)
    required = ['walk_model.zip', 'walk_model_vecnorm.pkl', 'walk_params.json']
    if walk_source_mode == 'drive_zip':
        bundle = Path(walk_bundle_zip)
        if not bundle.exists():
            raise FileNotFoundError(f'歩行ベースzipが見つかりません: {bundle}')
        shutil.unpack_archive(str(bundle), str(source_dir))
        print('Drive zipから歩行ベースを読み込みました:', bundle)
    elif walk_source_mode == 'repo_pretrained':
        source_dir = REPO / 'models/pretrained/smooth_walk'
        print('repo同梱のsmooth_walkを使います:', source_dir)
    else:
        raise ValueError(walk_source_mode)
    missing = [name for name in required if not (source_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f'歩行ベースに必要ファイルがありません: {missing} in {source_dir}')
    (BASE_DIR / 'walk_source_manifest.json').write_text(json.dumps({
        'walk_source_mode': walk_source_mode,
        'walk_bundle_zip': walk_bundle_zip,
        'resolved_source_dir': str(source_dir),
        'required': required,
        'prepared_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    return Path(source_dir)

WALK_SOURCE_DIR = prepare_walk_source()


def copy_walk_files(dst):
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    for name in ['walk_model.zip', 'walk_model_vecnorm.pkl', 'walk_params.json']:
        shutil.copy2(WALK_SOURCE_DIR / name, dst / name)

print('official-compatible helpers ready')
print('eval_seeds:', eval_seeds)


## 歩行モデルの単体確認

GAを回す前に、土台の `smooth_walk` がアリーナ環境で立つ・前進するか確認します。ここで0.2秒などで倒れる場合、逃げAIではなく歩行モデル/読み込み/環境互換の問題です。


In [ ]:
# 歩行モデルの単体確認
import numpy as np
from roboquest.envs.go2_tag_hierarchical_env import Go2TagHierarchicalEnv

walk_check_model = WALK_SOURCE_DIR / 'walk_model'
walk_check_vecnorm = WALK_SOURCE_DIR / 'walk_model_vecnorm.pkl'


def check_walk_command(action, label, seed_value=100, max_high_steps=150, required_seconds=None):
    env = Go2TagHierarchicalEnv(
        low_level_model_path=str(walk_check_model),
        low_level_vecnorm_path=str(walk_check_vecnorm),
        oni_speed=0.0,
    )
    try:
        obs, _ = env.reset(seed=seed_value)
        final_info = {}
        for step in range(max_high_steps):
            obs, reward, terminated, truncated, info = env.step(np.array(action, dtype=np.float32))
            final_info = info
            if terminated or truncated:
                break
        survived = float(final_info.get('survived_seconds', 0.0))
        tagged = bool(final_info.get('is_tagged', False))
        height = float(env.data.qpos[2])
        reason = 'running'
        if tagged:
            reason = 'tagged'
        elif survived < max_high_steps * 0.2:
            reason = 'fell_or_stopped_early'
        print(f'{label}: survived={survived:.1f}s, height={height:.3f}, tagged={tagged}, reason={reason}, cmd={final_info.get("velocity_command", action)}')
        ok = survived >= (required_seconds if required_seconds is not None else max_high_steps * 0.2)
        return survived, height, reason, ok
    finally:
        env.close()

print('歩行モデル単体チェックを開始します。鬼は停止させています。')
checks = [
    check_walk_command([0.0, 0.0, 0.0], 'stand', max_high_steps=150, required_seconds=20.0),
    # 高速直進を長く続けると5mアリーナの壁に当たるため、短時間の安定性だけ確認します。
    check_walk_command([1.0, 0.0, 0.0], 'forward_1.0', max_high_steps=15, required_seconds=3.0),
    check_walk_command([1.4, 0.0, 0.0], 'forward_1.4', max_high_steps=15, required_seconds=3.0),
]

if all(ok for _, _, _, ok in checks):
    print('✅ 土台の歩行は最低限OKです。逃げAIの学習に進めます。')
else:
    print('⚠️ 歩行モデルが単体で安定していません。GAを回す前に、モデル読み込みや環境差分を確認してください。')


In [ ]:
# 1個体の学習と評価（公式Tier1互換）
def train_and_eval(ind, generation, index):
    ind = normalize_individual(ind)
    cid = hash_individual(ind)
    folder = BASE_DIR / 'individuals' / cid
    done = folder / 'done.json'
    if done.exists():
        return json.loads(done.read_text(encoding='utf-8'))

    folder.mkdir(parents=True, exist_ok=True)
    copy_walk_files(folder)
    (folder / 'individual.json').write_text(json.dumps(ind, ensure_ascii=False, indent=2), encoding='utf-8')

    flee_cfg = FleeRewardConfig(
        survival_weight=float(ind['survival_weight']),
        distance_weight=float(ind['distance_weight']),
        tag_penalty=float(ind['tag_penalty']),
    )
    # 公式Tier1/quickstartのFLEE_PPOをそのまま使います。
    ppo = dict(FLEE_PPO)

    started = time.time()
    print(f"candidate {cid}: official_compatible=True, timesteps={int(ind['flee_timesteps'])}, envs={int(ind['flee_num_envs'])}, oni_speed={float(ind['oni_speed']):.4f}")
    print(f"  reward: survival={float(ind['survival_weight']):.3f}, distance={float(ind['distance_weight']):.3f}, tag_penalty={float(ind['tag_penalty']):.1f}")

    train_policy('flee', folder, flee_cfg, int(ind['flee_timesteps']), int(ind['flee_num_envs']),
                 ppo, seed=seed + generation * 1000 + index, oni_speed=float(ind['oni_speed']),
                 checkpoint_steps=50000, verbose=0, progress_bar=show_ppo_progress)

    results = evaluate_flee(folder, seeds=eval_seeds)
    summary = score_results(results)
    record = dict(candidate_id=cid, generation=generation, index=index, folder=str(folder),
                  official_default=(ind == normalize_individual(OFFICIAL_DEFAULT)),
                  individual=ind, results=results, summary=summary,
                  elapsed_seconds=time.time() - started)
    done.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
    append_jsonl(BASE_DIR / 'results.jsonl', record)

    csv_path = BASE_DIR / 'results.csv'
    exists = csv_path.exists()
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        fields = ['candidate_id','generation','index','official_default','score','mean_survived_seconds',
                  'worst_survived_seconds','escaped_count','tagged_count','fell_count','mean_distance','folder']
        w = csv.DictWriter(f, fieldnames=fields)
        if not exists:
            w.writeheader()
        row = {k: summary.get(k) for k in fields}
        row.update(candidate_id=cid, generation=generation, index=index,
                   official_default=(ind == normalize_individual(OFFICIAL_DEFAULT)), folder=str(folder))
        w.writerow(row)

    import gc
    gc.collect()
    return record

print('official-compatible train/eval ready')


In [ ]:
# GA実行（進捗バー付き）
from tqdm.auto import tqdm

started = time.time()
deadline = started + max_runtime_hours * 3600
state_path = BASE_DIR / 'state.json'
progress_path = BASE_DIR / 'progress_state.json'
if state_path.exists():
    state = json.loads(state_path.read_text(encoding='utf-8'))
    population = state['population']
    start_generation = state.get('generation', 0)
    best_record = state.get('best_record')
    print(f'既存runを再開します: generation={start_generation}')
else:
    population = make_initial_population()
    start_generation = 0
    best_record = None
    print('新規runを開始します。最初の個体は公式デフォルト設定です。')

remaining_total = max(0, generations - start_generation) * population_size
pbar = tqdm(total=remaining_total, desc='Tier1 GA candidates', unit='candidate')

for gen in range(start_generation, generations):
    print(f'=== Generation {gen+1}/{generations} ===')
    records = []
    for i, ind in enumerate(population):
        if time.time() > deadline - 1800:
            print('残り時間が少ないため、新規個体を開始せず保存して終了します。')
            state_path.write_text(json.dumps(dict(generation=gen, population=population, best_record=best_record), ensure_ascii=False, indent=2), encoding='utf-8')
            raise SystemExit
        cid = hash_individual(ind)
        progress_state = dict(status='running', generation=gen, generation_display=f'{gen+1}/{generations}',
                              individual_index=i, individual_display=f'{i+1}/{population_size}',
                              candidate_id=cid, started_at=time.strftime('%Y-%m-%d %H:%M:%S'),
                              profile=tier1_profile, run_name=run_name)
        progress_path.write_text(json.dumps(progress_state, ensure_ascii=False, indent=2), encoding='utf-8')
        pbar.set_description(f'Gen {gen+1}/{generations} Ind {i+1}/{population_size}')
        pbar.set_postfix(candidate=cid[:8], best=(None if best_record is None else round(best_record['summary']['score'], 2)))
        print(f'[Generation {gen+1}/{generations}] Individual {i+1}/{population_size} {cid}')
        rec = train_and_eval(ind, gen, i)
        records.append(rec)
        s = rec['summary']
        print(f"score={s['score']:.2f} mean={s['mean_survived_seconds']:.1f}s worst={s['worst_survived_seconds']:.1f}s escaped={s['escaped_count']} tagged={s['tagged_count']} fell={s['fell_count']}")
        if best_record is None or s['score'] > best_record['summary']['score']:
            best_record = rec
            best_dir = BASE_DIR / 'best_submission'
            best_dir.mkdir(parents=True, exist_ok=True)
            src = Path(rec['folder'])
            for name in ['walk_model.zip','walk_model_vecnorm.pkl','walk_params.json','flee_model.zip','flee_model_vecnorm.pkl','flee_params.json']:
                shutil.copy2(src / name, best_dir / name)
            (BASE_DIR / 'best_candidates.json').write_text(json.dumps(best_record, ensure_ascii=False, indent=2), encoding='utf-8')
            print('new best saved')
        progress_state.update(status='finished_candidate', last_score=s['score'],
                              last_mean_survived_seconds=s['mean_survived_seconds'],
                              best_score=(None if best_record is None else best_record['summary']['score']),
                              finished_at=time.strftime('%Y-%m-%d %H:%M:%S'))
        progress_path.write_text(json.dumps(progress_state, ensure_ascii=False, indent=2), encoding='utf-8')
        pbar.update(1)
    ranked = sorted(records, key=lambda r: r['summary']['score'], reverse=True)
    elites = [r['individual'] for r in ranked[:elite_count]]
    next_pop = list(elites)
    while len(next_pop) < population_size:
        a, b = rng.choice(elites), rng.choice(elites)
        next_pop.append(mutate(crossover(a, b)))
    population = next_pop
    state_path.write_text(json.dumps(dict(generation=gen+1, population=population, best_record=best_record), ensure_ascii=False, indent=2), encoding='utf-8')
    progress_path.write_text(json.dumps(dict(status='finished_generation', generation=gen+1,
                                             best_score=(None if best_record is None else best_record['summary']['score']),
                                             saved_at=time.strftime('%Y-%m-%d %H:%M:%S')),
                                      ensure_ascii=False, indent=2), encoding='utf-8')

pbar.close()
print('GA done')


## 進捗状況の確認

Driveに保存されているログを読み、現在の世代、評価済み個体数、ベストスコア、上位候補を表示します。学習中に別セルとして実行するより、学習セルが止まった後や再開前の確認に使うのが安全です。

In [ ]:
# 進捗状況の確認
import pandas as pd
from IPython.display import display

print(f'実験フォルダ: {BASE_DIR}')
state_path = BASE_DIR / 'state.json'
best_path = BASE_DIR / 'best_candidates.json'
results_path = BASE_DIR / 'results.csv'

if state_path.exists():
    state = json.loads(state_path.read_text(encoding='utf-8'))
    print(f"保存済み世代: {state.get('generation', 0)} / {generations}")
    print(f"次回再開時のpopulation数: {len(state.get('population', []))}")
else:
    print('state.json はまだありません。GA実行セルが世代末尾まで到達すると作成されます。')

if best_path.exists():
    best = json.loads(best_path.read_text(encoding='utf-8'))
    s = best.get('summary', {})
    print('\n現在のベスト')
    print(f" candidate_id: {best.get('candidate_id')}")
    print(f" score: {s.get('score', 0):.2f}")
    print(f" mean_survived_seconds: {s.get('mean_survived_seconds', 0):.1f}")
    print(f" worst_survived_seconds: {s.get('worst_survived_seconds', 0):.1f}")
    print(f" escaped/tagged/fell: {s.get('escaped_count', 0)} / {s.get('tagged_count', 0)} / {s.get('fell_count', 0)}")
    print(f" folder: {best.get('folder')}")
else:
    print('\nbest_candidates.json はまだありません。')

if results_path.exists():
    df = pd.read_csv(results_path)
    print(f'\n評価済み個体数: {len(df)}')
    if len(df):
        print(f"最新世代: {int(df['generation'].max())}")
        summary_cols = ['score','mean_survived_seconds','worst_survived_seconds','escaped_count','tagged_count','fell_count','mean_distance']
        existing_cols = [c for c in summary_cols if c in df.columns]
        
        base_cols = ['candidate_id','generation','index']
        if 'official_default' in df.columns:
            base_cols.append('official_default')
        display(df.sort_values('score', ascending=False).head(10)[base_cols + existing_cols + ['folder']])
        print('\n世代別サマリー')
        display(df.groupby('generation')[existing_cols].agg(['max','mean']).tail(10))
else:
    print('\nresults.csv はまだありません。')


## ベストモデルの動画確認

`best_submission/` に保存された現在のベストモデルを読み込み、俯瞰カメラの動画をDriveに保存します。学習セルとは別に、確認したい時だけ実行してください。

In [ ]:
# ベストモデルの動画確認
import imageio.v2 as imageio
from IPython.display import Video, display
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from roboquest.envs.go2_tag_hierarchical_env import Go2TagHierarchicalEnv
from roboquest.utils.reward_utils import FleeRewardConfig

video_seeds = [100, 101, 102] #@param
video_fps = 5 #@param {type:'integer'}
candidate_dir = BASE_DIR / 'best_submission'
video_dir = BASE_DIR / 'videos'
video_dir.mkdir(parents=True, exist_ok=True)

required = ['walk_model.zip','walk_model_vecnorm.pkl','flee_model.zip','flee_model_vecnorm.pkl','flee_params.json']
missing = [name for name in required if not (candidate_dir / name).exists()]
if missing:
    raise FileNotFoundError(f'best_submission に必要ファイルがありません: {missing}')

params = json.loads((candidate_dir / 'flee_params.json').read_text(encoding='utf-8'))
reward_cfg = FleeRewardConfig(**params['reward_config'])

def record_best(seed_value):
    raw_env_holder = {}
    def make_env():
        env = Go2TagHierarchicalEnv(
            low_level_model_path=str(candidate_dir / 'walk_model'),
            low_level_vecnorm_path=str(candidate_dir / 'walk_model_vecnorm.pkl'),
            flee_config=reward_cfg,
            oni_speed=float(params.get('oni_speed', 0.025)),
            render_mode='rgb_array',
        )
        raw_env_holder['env'] = env
        return env
    base = DummyVecEnv([make_env])
    vec = VecNormalize.load(str(candidate_dir / 'flee_model_vecnorm.pkl'), base)
    vec.training = False
    vec.norm_reward = False
    model = PPO.load(str(candidate_dir / 'flee_model'))
    frames = []
    try:
        vec.seed(int(seed_value))
        obs = vec.reset()
        final_info = {}
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, dones, infos = vec.step(action)
            frame = raw_env_holder['env'].render()
            if frame is not None:
                frames.append(frame.copy())
            if dones[0]:
                final_info = infos[0]
                break
    finally:
        vec.close()
    escaped = bool(final_info.get('TimeLimit.truncated', False))
    tagged = bool(final_info.get('is_tagged', False))
    survived = float(final_info.get('survived_seconds', 0.0))
    out = video_dir / f'best_seed{seed_value}_survived{survived:.1f}_escaped{int(escaped)}_tagged{int(tagged)}.mp4'
    imageio.mimsave(out, frames, fps=video_fps)
    print(f'seed={seed_value} survived={survived:.1f}s escaped={escaped} tagged={tagged} frames={len(frames)} -> {out}')
    return out

videos = [record_best(s) for s in video_seeds]
display(Video(str(videos[0]), embed=True))
